# RAG Spoofing — Exploratory Data Analysis
Run after `python -m src.pipeline.run_pipeline` (or at least after step 5).

In [ ]:
import json, re, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

def read_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

def read_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

## 1. Corpus statistics

In [ ]:
real_chunks  = read_jsonl('data/processed/corpus_chunks.jsonl')
spoof_chunks = read_jsonl('data/processed/spoof_chunks.jsonl')

real_lens  = [len(c['text'].split()) for c in real_chunks]
spoof_lens = [len(c['text'].split()) for c in spoof_chunks]

print(f'Real chunks : {len(real_chunks):,}  |  mean length: {np.mean(real_lens):.1f} words')
print(f'Spoof chunks: {len(spoof_chunks):,}  |  mean length: {np.mean(spoof_lens):.1f} words')

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(real_lens,  bins=30, color='#1D9E75', alpha=0.8, label='Real')
axes[0].hist(spoof_lens, bins=30, color='#D85A30', alpha=0.7, label='Spoof')
axes[0].set_xlabel('Chunk length (words)'); axes[0].set_ylabel('Count')
axes[0].set_title('Chunk length distribution'); axes[0].legend()

style_counts = Counter(c['attack_type'] for c in spoof_chunks)
axes[1].bar(style_counts.keys(), style_counts.values(), color='#534AB7', alpha=0.85)
axes[1].set_title('Spoof chunks per attack style')
axes[1].set_ylabel('Count'); plt.xticks(rotation=20, ha='right')
plt.tight_layout(); plt.show()

## 2. Lexical diversity — real vs spoof

In [ ]:
STOPWORDS = {'the','a','an','of','in','on','at','to','for','by','and','or','from','is','was'}

def ttr(text):
    words = [w for w in re.findall(r'\b\w+\b', text.lower()) if w not in STOPWORDS]
    return len(set(words)) / len(words) if words else 0

real_ttr  = [ttr(c['text']) for c in real_chunks[:2000]]
spoof_ttr = [ttr(c['text']) for c in spoof_chunks]

plt.figure(figsize=(7, 3.5))
plt.hist(real_ttr,  bins=30, color='#1D9E75', alpha=0.8, label=f'Real  (mean={np.mean(real_ttr):.2f})')
plt.hist(spoof_ttr, bins=30, color='#D85A30', alpha=0.7, label=f'Spoof (mean={np.mean(spoof_ttr):.2f})')
plt.xlabel('Type-token ratio (TTR)'); plt.ylabel('Count')
plt.title('Lexical diversity: real vs spoof chunks'); plt.legend()
plt.tight_layout(); plt.show()

## 3. Embedding space — real vs spoof (PCA)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

# Load base embeddings (real chunks) and augmented embeddings (real + spoof)
try:
    base_emb = np.load('indexes/minilm_base/embeddings.npy')
    atk_emb  = np.load('indexes/minilm_attack/embeddings.npy')
    n_real   = len(real_chunks)
    spoof_emb = atk_emb[n_real:]  # spoof embeddings appended after real

    sample_r = np.random.choice(len(base_emb),  min(500, len(base_emb)),  replace=False)
    sample_s = np.random.choice(len(spoof_emb), min(500, len(spoof_emb)), replace=False)
    combined = np.vstack([base_emb[sample_r], spoof_emb[sample_s]])
    labels   = ['real'] * len(sample_r) + ['spoof'] * len(sample_s)

    pca  = PCA(n_components=2)
    proj = pca.fit_transform(normalize(combined))

    plt.figure(figsize=(7, 5))
    mask_r = [l == 'real'  for l in labels]
    mask_s = [l == 'spoof' for l in labels]
    plt.scatter(proj[mask_r, 0], proj[mask_r, 1], c='#1D9E75', alpha=0.4, s=8, label='Real')
    plt.scatter(proj[mask_s, 0], proj[mask_s, 1], c='#D85A30', alpha=0.6, s=8, label='Spoof')
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    plt.title('PCA of MiniLM embeddings: real vs spoof')
    plt.legend(); plt.tight_layout(); plt.show()
except FileNotFoundError as e:
    print(f'Run the pipeline first: {e}')

## 4. Retrieval score distributions — baseline vs attack

In [ ]:
try:
    base_res = read_json('results/retrieval/minilm_baseline_results.json')
    atk_res  = read_json('results/retrieval/minilm_attack_results.json')

    base_scores  = [item['score'] for r in base_res.values() for item in r[:5]]
    atk_real     = [item['score'] for r in atk_res.values() for item in r[:5] if not item.get('is_spoof')]
    atk_spoof    = [item['score'] for r in atk_res.values() for item in r[:5] if item.get('is_spoof')]

    plt.figure(figsize=(8, 3.5))
    plt.hist(base_scores, bins=40, color='#1D9E75', alpha=0.7, label='Baseline real')
    plt.hist(atk_real,    bins=40, color='#534AB7', alpha=0.7, label='Attack — real chunks')
    plt.hist(atk_spoof,   bins=40, color='#D85A30', alpha=0.8, label='Attack — spoof chunks')
    plt.xlabel('Retrieval score (cosine similarity)')
    plt.ylabel('Count'); plt.title('Score distribution: top-5 retrieved chunks')
    plt.legend(); plt.tight_layout(); plt.show()

    print(f'Baseline real   mean score: {np.mean(base_scores):.4f}')
    print(f'Attack real     mean score: {np.mean(atk_real):.4f}')
    print(f'Attack spoof    mean score: {np.mean(atk_spoof):.4f}')
    print(f'Attraction margin (spoof - real): {np.mean(atk_spoof) - np.mean(atk_real):.4f}')
except FileNotFoundError as e:
    print(f'Run the pipeline first: {e}')

## 5. Defense suspicion score distribution

In [ ]:
try:
    def_res = read_json('results/retrieval/minilm_defense_results.json')
    real_sus  = [item['defense']['suspicion_score'] for r in def_res.values() for item in r if not item.get('is_spoof') and 'defense' in item]
    spoof_sus = [item['defense']['suspicion_score'] for r in def_res.values() for item in r if item.get('is_spoof') and 'defense' in item]

    plt.figure(figsize=(7, 3.5))
    plt.hist(real_sus,  bins=30, color='#1D9E75', alpha=0.8, label=f'Real  (mean={np.mean(real_sus):.2f})')
    plt.hist(spoof_sus, bins=30, color='#D85A30', alpha=0.7, label=f'Spoof (mean={np.mean(spoof_sus):.2f})')
    plt.axvline(0.45, color='black', linestyle='--', label='Threshold=0.45')
    plt.xlabel('Suspicion score'); plt.ylabel('Count')
    plt.title('Defense suspicion score: real vs spoof'); plt.legend()
    plt.tight_layout(); plt.show()
except FileNotFoundError as e:
    print(f'Run the pipeline first: {e}')